In [3]:
# 7a_synthetic_population.ipynb
#
# Builds the synthetic population parquet by combining three sources:
#
#   1. Load sipher_optimized.pkl            (synthetic_zone, pidp)
#   2. Map each LSOA (synthetic_zone) to MSOA and Local Authority
#   3. Join UKHLS rows (4_feature_eng/{WAVE}_feature_eng.pkl from steps 4a–6) on pidp — inner join so every
#      output row has UKHLS demographics (sipher pidp not in the feature table are dropped).
#   4. Stream-write to parquet in chunks    (avoids holding full dataset in RAM)

import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from data_pipeline.helpers.pipeline_checks import warn_non_numeric
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from data_pipeline.config_variables import DATA_FOLDER, WAVE

# ── Config ────────────────────────────────────────────────────────────────────
SIPHER_PKL  = f"../{DATA_FOLDER}/1_pickle_sipher/sipher_optimized.pkl"
FEATURE_PKL = f"../{DATA_FOLDER}/4_feature_eng/{WAVE}_feature_eng.pkl"
GEO_CSV     = f"../{DATA_FOLDER}/0_raw/admin_geography_mappings.csv"
OUTPUT_FILE = Path(f"../{DATA_FOLDER}/7_synthetic_population/synthetic_population.parquet")
CHUNK_SIZE  = 500_000   # rows of sipher data processed per iteration

for p in [SIPHER_PKL, FEATURE_PKL, GEO_CSV]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"{p} not found — check pipeline prerequisites.")

# ── Load lookup tables (small — kept in memory throughout) ───────────────────
print("Loading lookup tables ...")

geo_cols = ['lsoa21cd', 'msoa21cd', 'msoa21nm', 'ladcd', 'ladnm']
df_geo = (
    pd.read_csv(GEO_CSV, usecols=geo_cols, dtype=str, encoding='latin-1')
    .drop_duplicates(subset='lsoa21cd')
    .set_index('lsoa21cd')
)
print(f"  Geography:  {len(df_geo):,} unique LSOAs")

df_features = pd.read_pickle(FEATURE_PKL)
df_features['pidp'] = df_features['pidp'].astype('int64')
df_features = df_features.set_index('pidp')
print(f"  Features:   {len(df_features):,} UKHLS respondents × {len(df_features.columns)} cols")

# ── Load sipher index only (to determine chunks) ──────────────────────────────
print("\nLoading sipher pickle ...")
df_sipher = pd.read_pickle(SIPHER_PKL)[['synthetic_zone', 'pidp']].copy()
df_sipher['pidp'] = df_sipher['pidp'].astype('int64')
df_sipher.sort_values('synthetic_zone', inplace=True)
df_sipher.reset_index(drop=True, inplace=True)
n_total = len(df_sipher)
n_chunks = (n_total + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"  {n_total:,} rows — processing in {n_chunks} chunk(s) of {CHUNK_SIZE:,}")

# ── Stream-write parquet chunk by chunk ───────────────────────────────────────
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
writer = None
n_no_geo = 0
n_unmatched = 0
n_written = 0

for i in range(n_chunks):
    start = i * CHUNK_SIZE
    end   = min(start + CHUNK_SIZE, n_total)
    chunk = df_sipher.iloc[start:end].copy()

    # 2. Map LSOA → MSOA / LA
    chunk = chunk.join(df_geo, on='synthetic_zone', how='left')
    n_no_geo += int(chunk['ladcd'].isna().sum())

    # 3. Join UKHLS features (inner: drop sipher rows whose pidp is not in UKHLS extracts)
    n_before = len(chunk)
    chunk = chunk.join(df_features, on='pidp', how='inner')
    n_unmatched += n_before - len(chunk)

    if len(chunk) == 0:
        print(f"  Chunk {i+1}/{n_chunks}: no rows — skip")
        continue

    # Write chunk
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_FILE, table.schema, compression='snappy')
    writer.write_table(table)
    n_written += len(chunk)

    print(f"  Chunk {i+1}/{n_chunks}: rows {start:,}–{end:,} written")
    del chunk, table
    gc.collect()

if writer:
    writer.close()

del df_sipher, df_geo, df_features
gc.collect()

# ── Summary ───────────────────────────────────────────────────────────────────
size_mb = OUTPUT_FILE.stat().st_size / 1e6
if n_no_geo:
    print(f"\nWarning: {n_no_geo:,} rows ({100*n_no_geo/n_total:.2f}%) could not be mapped to a geography")
if n_unmatched:
    print(
        f"Note: {n_unmatched:,} sipher rows ({100*n_unmatched/n_total:.2f}%) dropped — "
        f"pidp not in UKHLS feature table (inner join)."
    )

print(f"\nDone.")
print(f"  Rows written:  {n_written:,}  (sipher input rows: {n_total:,})")
print(f"  Disk:  {size_mb:.0f} MB")
print(f"  File:  {OUTPUT_FILE}")

if n_written > 0:
    _geo_strings = frozenset({"synthetic_zone", "msoa21cd", "msoa21nm", "ladcd", "ladnm"})
    warn_non_numeric(
        pd.read_parquet(OUTPUT_FILE),
        "6a_synthetic_population",
        ignore_cols=_geo_strings,
    )
else:
    print("\n[6a_synthetic_population] Skipping non-numeric audit — no rows written.")


Loading lookup tables ...
  Geography:  43,501 unique LSOAs
  Features:   19,618 UKHLS respondents × 62 cols

Loading sipher pickle ...
  52,853,971 rows — processing in 106 chunk(s) of 500,000
  Chunk 1/106: no rows in selected LAs — skip
  Chunk 2/106: no rows in selected LAs — skip
  Chunk 3/106: no rows in selected LAs — skip
  Chunk 4/106: no rows in selected LAs — skip
  Chunk 5/106: no rows in selected LAs — skip
  Chunk 6/106: no rows in selected LAs — skip
  Chunk 7/106: no rows in selected LAs — skip
  Chunk 8/106: rows 3,500,000–4,000,000 written
  Chunk 9/106: no rows in selected LAs — skip
  Chunk 10/106: rows 4,500,000–5,000,000 written
  Chunk 11/106: rows 5,000,000–5,500,000 written
  Chunk 12/106: rows 5,500,000–6,000,000 written
  Chunk 13/106: rows 6,000,000–6,500,000 written
  Chunk 14/106: no rows in selected LAs — skip
  Chunk 15/106: no rows in selected LAs — skip
  Chunk 16/106: no rows in selected LAs — skip
  Chunk 17/106: no rows in selected LAs — skip
  Chun